# 04 – Autoencoder

PyTorch Autoencoder: 20→64→32→12→32→64→20 (50 epochs, Adam lr=1e-3).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
from src.utils.preprocessing import generate_synthetic_dataset, preprocess_pipeline, FEATURE_COLUMNS
from src.models.autoencoder import SatelliteAutoencoder
df = generate_synthetic_dataset(n_samples=120_000, seed=42)
feature_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
train, val, test, scaler, _ = preprocess_pipeline(df, feature_cols)
X_train = train[feature_cols].values; X_val = val[feature_cols].values
X_test = test[feature_cols].values; y_test = test['label'].values

In [ ]:
model = SatelliteAutoencoder(input_dim=len(feature_cols), epochs=50)
model.fit(X_train, X_val)
metrics = model.evaluate(X_test, y_test)
for k, v in metrics.items(): print(f'  {k}: {v:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model.train_losses_, label='Train')
if model.val_losses_: ax.plot(model.val_losses_, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.savefig('../results/ae_training_curve.png', dpi=100); plt.show()

In [ ]:
scores = model.anomaly_scores(X_test)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores[y_test==0], bins=60, alpha=0.6, label='Normal', color='steelblue')
ax.hist(scores[y_test==1], bins=60, alpha=0.6, label='Anomaly', color='red')
ax.axvline(model.threshold_, color='orange', linestyle='--', label=f'Threshold ({model.threshold_:.4f})')
ax.set_xlabel('Reconstruction Error'); ax.legend()
plt.tight_layout(); plt.savefig('../results/reconstruction_error_hist.png', dpi=100); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, scores, ax=ax, name='Autoencoder')
plt.tight_layout(); plt.savefig('../results/roc_curve.png', dpi=100); plt.show()